# 3.4. Requerimientos

Subsección 3.4 de CRISP-DM: respuesta a los 9 requerimientos de información del cliente, cada uno con visualización y tabla de datos.

In [ ]:
import json, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

DATA_DIR = '../data'
CSV_PATH = f'{DATA_DIR}/GBvideos_cc50_202101.csv'
JSON_PATH = f'{DATA_DIR}/GB_category_id.json'
FIG_DIR = '../reports/figures'
os.makedirs(FIG_DIR, exist_ok=True)

# Carga, mapeo de categorías y variables derivadas necesarias para el negocio
df = pd.read_csv(CSV_PATH)
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    categories_data = json.load(f)
category_map = {int(i['id']): i['snippet']['title'] for i in categories_data['items']}
df['category_name'] = df['category_id'].map(category_map).fillna('Unknown')
df['trending_date'] = pd.to_datetime(df['trending_date'], errors='coerce')

df['like_dislike_ratio'] = df['likes'] / (df['dislikes'] + 1)
df['views_per_comment'] = df['views'] / (df['comment_count'] + 1)
df['engagement_rate'] = (df['likes'] + df['dislikes'] + df['comment_count']) / (df['views'] + 1)
print('Shape:', df.shape)

## 3.4.1 ¿Qué categorías de videos son las de mayor tendencia?

In [ ]:
top_cats = df['category_name'].value_counts().head(10)
print(top_cats)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_cats.values, y=top_cats.index, palette='viridis',
            hue=top_cats.index, legend=False)
plt.title('3.4.1 - Top 10 categorías en tendencias')
plt.xlabel('Frecuencia')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/business_p1_categories.png', dpi=300, bbox_inches='tight'); plt.show()
display(top_cats.to_frame('frecuencia'))

## 3.4.2 ¿Qué categorías gustan más? ¿Y cuáles menos?

In [ ]:
likes_by_cat = df.groupby('category_name')['likes'].agg(['mean', 'median', 'count']).sort_values('mean', ascending=False)
mas = likes_by_cat.head(5); menos = likes_by_cat.tail(5)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(x=mas['mean'], y=mas.index, palette='Greens_r', hue=mas.index, legend=False, ax=axes[0])
axes[0].set_title('3.4.2 - Categorías que MÁS gustan'); axes[0].set_xlabel('Likes promedio')
sns.barplot(x=menos['mean'], y=menos.index, palette='Reds', hue=menos.index, legend=False, ax=axes[1])
axes[1].set_title('3.4.2 - Categorías que MENOS gustan'); axes[1].set_xlabel('Likes promedio')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/business_p2_likes.png', dpi=300, bbox_inches='tight'); plt.show()
display(likes_by_cat)

## 3.4.3 ¿Qué categorías tienen la mejor proporción Me gusta / No me gusta?

In [ ]:
ratio_by_cat = (
    df.groupby('category_name')['like_dislike_ratio']
    .agg(['mean', 'median'])
    .sort_values('mean', ascending=False)
)
top_ratio = ratio_by_cat.head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_ratio['mean'], y=top_ratio.index, palette='crest',
            hue=top_ratio.index, legend=False)
plt.title('3.4.3 - Top 10 categorías por ratio Likes/Dislikes')
plt.xlabel('Ratio likes/dislikes (promedio)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/business_p3_ratio_likes_dislikes.png', dpi=300, bbox_inches='tight'); plt.show()
display(ratio_by_cat)

## 3.4.4 ¿Qué categorías tienen la mejor proporción Vistas / Comentarios?

In [ ]:
vc_ratio = (
    df.groupby('category_name')['views_per_comment']
    .agg(['mean', 'median'])
    .sort_values('mean', ascending=False)
)
top_vc = vc_ratio.head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_vc['mean'], y=top_vc.index, palette='flare',
            hue=top_vc.index, legend=False)
plt.title('3.4.4 - Top 10 categorías por ratio Vistas/Comentarios')
plt.xlabel('Vistas por comentario (promedio)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/business_p4_ratio_views_comments.png', dpi=300, bbox_inches='tight'); plt.show()
display(vc_ratio)

## 3.4.5 ¿Cómo ha cambiado el volumen de videos en tendencia a lo largo del tiempo?

In [ ]:
serie = df.groupby(df['trending_date'].dt.to_period('W'))['views'].mean().reset_index()
serie['trending_date'] = serie['trending_date'].dt.to_timestamp()
serie['media_movil'] = serie['views'].rolling(4, min_periods=1).mean()

plt.figure(figsize=(12, 5))
sns.lineplot(data=serie, x='trending_date', y='views', label='Vistas prom.', color='navy')
sns.lineplot(data=serie, x='trending_date', y='media_movil', label='Media móvil (4)', color='red')
plt.title('3.4.5 - Evolución temporal de vistas')
plt.legend()
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/business_p5_trend.png', dpi=300, bbox_inches='tight'); plt.show()

## 3.4.6 ¿Qué canales de YouTube son tendencia más frecuentemente? ¿Y cuáles con menos frecuencia?

In [ ]:
channel_counts = df['channel_title'].value_counts()
top_channels = channel_counts.head(10)
menos_frecuentes = channel_counts[channel_counts == channel_counts.min()]
print(f'Canales con solo {channel_counts.min()} aparición: {len(menos_frecuentes)}')
print('Ejemplos:', menos_frecuentes.head(10).index.tolist())

plt.figure(figsize=(10, 6))
sns.barplot(x=top_channels.values, y=top_channels.index, palette='mako',
            hue=top_channels.index, legend=False)
plt.title('3.4.6 - Top 10 canales más frecuentes en tendencias')
plt.xlabel('N.º de apariciones')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/business_p6_channels.png', dpi=300, bbox_inches='tight'); plt.show()
display(top_channels.to_frame('apariciones'))

## 3.4.7 ¿En qué Estados se presenta el mayor número de Vistas, Me gusta y No me gusta?

In [ ]:
geo = df.groupby('state').agg(
    videos=('video_id', 'count'),
    views=('views', 'sum'),
    likes=('likes', 'sum'),
    dislikes=('dislikes', 'sum'),
).sort_values('views', ascending=False)
display(geo.head(15))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, color in zip(axes, ['views', 'likes', 'dislikes'], ['Blues_r', 'Greens_r', 'Reds_r']):
    top = geo.sort_values(metric, ascending=False).head(10)
    sns.barplot(x=top[metric], y=top.index, palette=color, hue=top.index, legend=False, ax=ax)
    ax.set_title(f'3.4.7 - Top 10 estados por {metric}')
    ax.set_xlabel(f'Total {metric}')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/business_p7_states.png', dpi=300, bbox_inches='tight'); plt.show()

try:
    import folium
    from folium.plugins import HeatMap
    m = folium.Map(location=[54, -2], zoom_start=5)
    HeatMap(df[['lat', 'lon', 'views']].dropna().values.tolist(), radius=10).add_to(m)
    m.save(f'{FIG_DIR}/business_p7_heatmap.html')
    print('Mapa guardado en', f'{FIG_DIR}/business_p7_heatmap.html')
except ImportError:
    print('folium no instalado; se usan las barras por estado.')

## 3.4.8 ¿Los videos en tendencia son los que mayor cantidad de comentarios positivos reciben?

In [ ]:
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    analyzer = SentimentIntensityAnalyzer()
    df['sentiment_compound'] = df['title'].astype(str).apply(
        lambda t: analyzer.polarity_scores(t)['compound'])
except ImportError:
    from textblob import TextBlob
    df['sentiment_compound'] = df['title'].astype(str).apply(
        lambda t: TextBlob(t).sentiment.polarity)

df['sentiment_label'] = pd.cut(
    df['sentiment_compound'],
    bins=[-1.01, -0.05, 0.05, 1.01],
    labels=['negative', 'neutral', 'positive'],
)
print(df['sentiment_label'].value_counts(normalize=True).round(3))

### 3.4.8.1 Relación entre sentimiento del título y vistas

In [ ]:
sent_views = df.groupby('sentiment_label', observed=True)['views'].agg(['mean', 'median', 'count'])
display(sent_views)

plt.figure(figsize=(8, 5))
sns.barplot(x=sent_views.index, y=sent_views['mean'], palette='Set2',
            hue=sent_views.index, legend=False)
plt.title('3.4.8 - Vistas promedio según sentimiento del título')
plt.ylabel('Vistas promedio'); plt.xlabel('Sentimiento')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/business_p8_sentiment.png', dpi=300, bbox_inches='tight'); plt.show()

print('Nota: el dataset no incluye el texto de los comentarios, por lo que se usa el')
print('sentimiento del TÍTULO como proxy de la percepción/recepción del video.')

## 3.4.9 ¿Es factible predecir el número de Vistas, Me gusta o No me gusta?

In [ ]:
MODEL_CSV = f'{DATA_DIR}/processed/gb_videos_model.csv'
if os.path.exists(MODEL_CSV):
    model_df = pd.read_csv(MODEL_CSV)
    print('Dataset de modelado listo:', model_df.shape)
else:
    print('Ejecuta 05_preparacion_datos.ipynb y 07_modelado_evaluacion.ipynb para generar el modelo.')

print('La factibilidad completa se evalúa en 07_modelado_evaluacion.ipynb y en 09_verificacion_cumplimiento.ipynb.')